# Feature Engineering — Machine Learning Analysis

## Introduction

Feature engineering is the process of preparing and transforming variables so that they can be used more effectively by a machine learning model.

In this activity, different feature-engineering techniques are explored and their effect on a machine learning workflow is examined. The focus is on creating useful representations of the available variables while keeping the modelling process clear and reproducible.

The analysis also considers how transformations can affect model performance and why preprocessing should be handled carefully.


## Objectives

The main objectives of this activity are:

- To understand the purpose of feature engineering.
- To inspect the available variables before transformation.
- To create or transform features where appropriate.
- To compare the data before and after feature engineering.
- To use the engineered features in a machine learning workflow.
- To evaluate whether the transformations improve model performance.


## 1. Importing the Required Libraries

The libraries below are used for data handling, visualisation, feature preparation and model evaluation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully.")


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_selection import SelectKBest, f_regression

data = load_diabetes(as_frame=True)
df = data.frame.copy()
display(df.head())


### Observation

The output above is used to check the effect of this step before moving to the next part of the analysis. The final interpretation should be based on the values produced when the notebook is executed.


### Analysis Step

The following cell continues the feature-engineering workflow.

In [ ]:
# Inspect distributions and correlations
display(df.describe().T)

plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()


## 4. Feature Engineering

The selected variables are transformed or combined to create a representation suitable for modelling.

In [ ]:
# Example feature engineering
eps = 1e-6
df_fe = df.copy()

df_fe["bmi_squared"] = df_fe["bmi"] ** 2
df_fe["bp_squared"] = df_fe["bp"] ** 2
df_fe["bmi_bp_interaction"] = df_fe["bmi"] * df_fe["bp"]
df_fe["bmi_age_ratio"] = df_fe["bmi"] / (np.abs(df_fe["age"]) + eps)
df_fe["s5_log_abs"] = np.log1p(np.abs(df_fe["s5"]))

display(df_fe.head())


## 2. Loading and Inspecting the Data

The dataset is loaded and inspected before any feature transformations are applied.

In [ ]:
# Polynomial features
X = df[["bmi", "bp", "age"]]
y = df["target"]

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

poly_df = pd.DataFrame(X_poly, columns=poly.get_feature_names_out(X.columns))
display(poly_df.head())
print("Original feature count:", X.shape[1])
print("Polynomial feature count:", X_poly.shape[1])


In [ ]:
# Quantile/binning example
df_bins = df.copy()
df_bins["bmi_group"] = pd.qcut(df_bins["bmi"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
display(df_bins[["bmi", "bmi_group", "target"]].head(10))


## 3. Preparing the Data for Modelling

The predictors and target are prepared before the train-test split is created.

In [ ]:
# Compare baseline vs engineered polynomial model
X_base = df.drop(columns="target")
X_eng = df_fe.drop(columns="target")

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)
Xe_train, Xe_test, ye_train, ye_test = train_test_split(
    X_eng, y, test_size=0.2, random_state=42
)

baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10))
])
engineered = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10))
])

baseline.fit(Xb_train, yb_train)
engineered.fit(Xe_train, ye_train)

base_pred = baseline.predict(Xb_test)
eng_pred = engineered.predict(Xe_test)

display(pd.DataFrame({
    "Model": ["Baseline", "Engineered"],
    "RMSE": [
        mean_squared_error(yb_test, base_pred) ** 0.5,
        mean_squared_error(ye_test, eng_pred) ** 0.5
    ],
    "R2": [
        r2_score(yb_test, base_pred),
        r2_score(ye_test, eng_pred)
    ]
}))


In [ ]:
# Feature selection
X = df.drop(columns="target")
selector = SelectKBest(score_func=f_regression, k=5)
X_selected = selector.fit_transform(X, y)

selected_names = X.columns[selector.get_support()]
print("Selected features:", list(selected_names))
display(pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False))


## 7. Overall Interpretation

Feature engineering is useful when the original variables do not provide the most suitable representation for a machine learning model. The transformations used in this activity should therefore be judged by whether they make the modelling process more effective rather than by adding complexity alone.


## 8. Conclusion

This activity demonstrated how feature engineering can be incorporated into a machine learning workflow. The data was inspected, transformed where required, and then used for modelling and evaluation.

The results from the executed notebook should be used to determine whether the engineered representation provides a meaningful improvement for the model tested.


## 9. Practice Questions

The practice questions from the activity are retained below so that the required assessment component remains part of the notebook.


## Practice questions

- Why can a ratio feature be more useful than its original variables?
- What is the danger of generating too many polynomial features?
- Why should feature engineering be performed consistently on train and test data?
- Which engineered features improved your model and why?
